In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/turkish_law_dataset.csv")

In [3]:
df.head()

,soru,cevap,veri türü,kaynak,context,Score
0,"Anayasa, Türk Vatanı ve Milletinin ebedi varlı...","Anayasa, Türk Vatanı ve Milletinin ebedi varlı...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
1,"Anayasa, Türkiye Cumhuriyetinin hangi milliyet...","Anayasa, Türkiye Cumhuriyetinin kurucusu olan ...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,8
2,"Anayasa, Türkiye Cumhuriyetini hangi konumda t...","Anayasa, Türkiye Cumhuriyetini dünya milletler...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
3,"Anayasa, Türkiye Cumhuriyetinin hangi hedefler...","Anayasa, Türkiye Cumhuriyetinin ebedi varlığın...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,9
4,"Anayasa, egemenliğin kime ait olduğunu nasıl b...","Anayasa, egemenliğin kayıtsız şartsız Türk Mil...",hukuk,Türkiye Cumhuriyeti Anayasası,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,10


In [4]:
df.shape

(13954, 6)

In [5]:
df.columns.tolist()

['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'Score']

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13954 entries, 0 to 13953
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   soru       13954 non-null  object
 1   cevap      13954 non-null  object
 2   veri türü  13954 non-null  object
 3   kaynak     13954 non-null  object
 4   context    13954 non-null  object
 5   Score      13954 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 654.2+ KB


In [7]:
df.isnull().sum()

soru         0
cevap        0
veri türü    0
kaynak       0
context      0
Score        0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(247)

In [9]:
df.sample(3, random_state=42)

,soru,cevap,veri türü,kaynak,context,Score
2019,Adlî kontrol hükümlerini yerine getirmeyen şüp...,Adlî kontrol hükümlerini isteyerek yerine geti...,hukuk,Ceza Muhakemesi Kanunu,BİRİNCİ KİTAP\r\nGenel Hükümler\r\nDÖRDÜNCÜ KI...,8
9438,Ceza kanunlarını bilmemek hangi durumlarda maz...,Ceza kanunlarını bilmemek genellikle mazeret s...,hukuk,Türk Ceza Kanunu,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,8
5709,"Birisi yaşadığı yerin dışında öldüğünde, ölüm ...","Birisi yaşadığı yerin dışında öldüğünde, ölüm ...",hukuk,Türk Medeni Kanunu,ÜÇÜNCÜ KİTAP\r\nMİRAS HUKUKU\r\nİKİNCİ KISIM\r...,8


In [10]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=["soru", "cevap", "context"])

df_clean = df_clean.drop_duplicates()

print("New shape:", df_clean.shape)

New shape: (13707, 6)


In [11]:
df_clean["soru_len"] = df_clean["soru"].astype(str).apply(len)
df_clean["cevap_len"] = df_clean["cevap"].astype(str).apply(len)
df_clean["context_len"] = df_clean["context"].astype(str).apply(len)

df_clean = df_clean[
    (df_clean["soru_len"] > 10) &
    (df_clean["cevap_len"] > 10) &
    (df_clean["context_len"] > 50)
]

print("Shape after filtering:", df_clean.shape)

Shape after filtering: (13707, 9)


In [12]:
df_clean = df_clean.drop_duplicates(subset=["soru"])

print("After question-based dedup:", df_clean.shape)

After question-based dedup: (12885, 9)


In [13]:
retrieval_df = df_clean[["context"]].copy()

retrieval_df = retrieval_df.drop_duplicates()

print("Retrieval dataset size:", retrieval_df.shape)

Retrieval dataset size: (240, 1)


In [14]:
def chunk_text(text, chunk_size=300, overlap=50):
    text = str(text)
    chunks = []
    
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
        
    return chunks

In [15]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = chunk_text(text)
    all_chunks.extend(chunks)

print("Total chunks:", len(all_chunks))

Total chunks: 5811


In [16]:
chunks_df = pd.DataFrame({"chunk_text": all_chunks})
chunks_df = chunks_df.drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head()

Unique chunks: (5811, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...
1,ılap ve ilkeleri doğrultusunda;\n\nDünya mille...
2,"lak üstünlüğü, egemenliğin kayıtsız şartsız Tü..."
3,"cağı;\n\nKuvvetler ayrımının, Devlet organları..."
4,da bulunduğu;\n\nHiçbir faaliyetin Türk milli ...


In [17]:
chunks_df["chunk_len"] = chunks_df["chunk_text"].astype(str).apply(len)
chunks_df["chunk_len"].describe()

count    5811.000000
mean      292.358458
std        38.708697
min         1.000000
25%       300.000000
50%       300.000000
75%       300.000000
max       300.000000
Name: chunk_len, dtype: float64

In [18]:
chunks_df.to_csv("../data/processed/retrieval_chunks.csv", index=False, encoding="utf-8-sig")

print("Saved!")

Saved!


In [19]:
print(chunks_df.shape)
chunks_df.head()

(5811, 2)


,chunk_text,chunk_len
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,300
1,ılap ve ilkeleri doğrultusunda;\n\nDünya mille...,300
2,"lak üstünlüğü, egemenliğin kayıtsız şartsız Tü...",300
3,"cağı;\n\nKuvvetler ayrımının, Devlet organları...",300
4,da bulunduğu;\n\nHiçbir faaliyetin Türk milli ...,300


In [20]:
chunks_df = pd.read_csv("../data/processed/retrieval_chunks.csv")
print(chunks_df.shape)
chunks_df.head()

(5811, 2)


,chunk_text,chunk_len
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,300
1,ılap ve ilkeleri doğrultusunda;\n\nDünya mille...,300
2,"lak üstünlüğü, egemenliğin kayıtsız şartsız Tü...",300
3,"cağı;\n\nKuvvetler ayrımının, Devlet organları...",300
4,da bulunduğu;\n\nHiçbir faaliyetin Türk milli ...,300


In [21]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

c:\Users\Gaming\Desktop\turkish-legal-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3184.85it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
sample_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].head(5).tolist(),
    show_progress_bar=True
)

print(sample_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]

(5, 384)


In [23]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/182 [00:00<?, ?it/s]

Batches: 100%|██████████| 182/182 [02:31<00:00,  1.20it/s]

Embeddings shape: (5811, 384)


In [24]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index total vectors:", index.ntotal)

FAISS index total vectors: 5811


In [25]:
query = "Türkiye Cumhuriyetinin yönetim şekli nedir?"
query_embedding = embedding_model.encode([query], convert_to_numpy=True)

k = 5
distances, indices = index.search(query_embedding, k)

print("Top 5 retrieved chunks:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"{rank}. Sonuç:\n")
    print(chunks_df.iloc[idx]["chunk_text"])
    print("\n" + "-" * 100 + "\n")

Top 5 retrieved chunks:

1. Sonuç:

ve tarih huzurunda, namusum ve şerefim üzerine andiçerim."

 

D. Görev ve yetkileri

Madde 104  – (Değişik: 21/1/2017-6771/8 md.)

Cumhurbaşkanı Devletin başıdır. Yürütme yetkisi Cumhurbaşkanına aittir.

Cumhurbaşkanı, Devlet başkanı sıfatıyla Türkiye Cumhuriyetini ve Türk Milletinin birliğini tems

----------------------------------------------------------------------------------------------------

2. Sonuç:

ÜÇÜNCÜ KISIM

CUMHURİYETİN TEMEL ORGANLARI

BİRİNCİ BÖLÜM

Yasama

Öncesi…

II. Türkiye Büyük Millet Meclisinin görev ve yetkileri

 

A. Genel olarak

Madde 87 – (Değişik: 21/1/2017-6771/5 md.)

Türkiye Büyük Millet Meclisinin görev ve yetkileri, kanun koymak, değiştirmek ve kaldırmak; bütçe ve kes

----------------------------------------------------------------------------------------------------

3. Sonuç:

BAŞLANGIÇ [5]

 

Türk Vatanı ve Milletinin ebedi varlığını ve Yüce Türk Devletinin bölünmez bütünlüğünü belirleyen bu Anayasa, Türkiy

In [26]:
print(indices)
print(indices.shape)

[[ 57  10   0 318   1]]
(1, 5)


In [27]:
def retrieve_top_k(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(dist)
        })
    return results

In [28]:
results = retrieve_top_k(
    "Egemenlik kime aittir?",
    embedding_model,
    index,
    chunks_df,
    k=3
)

for i, item in enumerate(results, 1):
    print(f"{i}. Distance: {item['distance']}")
    print(item["chunk_text"])
    print("-" * 100)

1. Distance: 11.892723083496094
lak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;

Kuvvetler ayrımının, Devlet organları arası
----------------------------------------------------------------------------------------------------
2. Distance: 12.940750122070312

kendi başına yönetmek veya bunun için temsilci atamak gücünden yoksunsa,
3. Bir terekede mirasçılık hakları henüz belli değilse veya ceninin menfaatleri gerekli kılarsa, 
4. Bir tüzel kişi gerekli organlardan yoksun kalmış ve yönetimi başka yoldan 
sağlanamamışsa,
5. Bir hayır işi veya genel ya
----------------------------------------------------------------------------------------------------
3. Distance: 13.368047714233398
İKİNCİ KISIM

TEMEL HAKLAR VE ÖDEVLER

 

BİRİNCİ BÖLÜM

Genel Hükümler

 

I.  Temel hak ve hürriyetler

In [29]:
def chunk_text_smart(text, chunk_size=300, overlap=50):
    text = str(text).strip()
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)

        if end < text_len:
            while end > start and text[end] != " ":
                end -= 1
            if end == start:
                end = min(start + chunk_size, text_len)

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        next_start = end - overlap
        if next_start <= start:
            next_start = end

        if next_start < text_len and next_start > 0:
            while next_start < text_len and text[next_start] != " ":
                next_start += 1
            while next_start < text_len and text[next_start] == " ":
                next_start += 1

        start = next_start

    return chunks

In [30]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = chunk_text_smart(text)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head()

Unique chunks: (5965, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...
1,ve ilkeleri doğrultusunda;\n\nDünya milletleri...
2,"üstünlüğü, egemenliğin kayıtsız şartsız Türk M..."
3,"ayrımının, Devlet organları arasında üstünlük ..."
4,"faaliyetin Türk milli menfaatlerinin, Türk var..."


In [31]:
import re

def split_long_text_safely(text, max_len=300, overlap=50):
    text = str(text).strip()
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + max_len, n)

        if end < n:
            split_pos = text.rfind(" ", start, end)
            if split_pos == -1 or split_pos <= start:
                split_pos = end
        else:
            split_pos = end

        chunk = text[start:split_pos].strip()
        if chunk:
            chunks.append(chunk)

        next_start = split_pos - overlap
        if next_start <= start:
            next_start = split_pos

        start = next_start

    return chunks


def chunk_text_by_paragraph(text, max_len=300, overlap=50, min_len=80):
    text = str(text).replace("\\n", "\n").strip()

    parts = re.split(r"\n\s*\n+", text)
    parts = [p.strip() for p in parts if p.strip()]

    merged_parts = []
    buffer = ""

    for part in parts:
        if len(buffer) == 0:
            buffer = part
        elif len(buffer) + len(part) + 2 < min_len:
            buffer += "\n\n" + part
        else:
            merged_parts.append(buffer)
            buffer = part

    if buffer:
        merged_parts.append(buffer)

    final_chunks = []

    for part in merged_parts:
        if len(part) <= max_len:
            final_chunks.append(part)
        else:
            final_chunks.extend(split_long_text_safely(part, max_len=max_len, overlap=overlap))

    return final_chunks

In [32]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = chunk_text_by_paragraph(text, max_len=300, overlap=50, min_len=80)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head(10)

Unique chunks: (7680, 1)


,chunk_text
0,BAŞLANGIÇ [5]
1,Türk Vatanı ve Milletinin ebedi varlığını ve Y...
2,Dünya milletleri ailesinin eşit haklara sahip ...
3,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
4,"Kuvvetler ayrımının, Devlet organları arasında..."
5,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
6,ceği ve laiklik ilkesinin gereği olarak kutsal...
7,ve politikaya kesinlikle karıştırılamayacağı; [6]
8,Her Türk vatandaşının bu Anayasadaki temel hak...
9,Topluca Türk vatandaşlarının milli gurur ve if...


In [33]:
import re
import pandas as pd

BAD_START_WORDS = {
    "ve", "veya", "ile", "da", "de", "ama", "fakat", "ancak",
    "gibi", "çünkü", "ise", "ya", "ya da", "hem", "ki"
}

def starts_bad(text):
    text = str(text).strip().lower()
    if not text:
        return True
    
    first_words = text.split()[:2]
    if not first_words:
        return True
    
    first_word = first_words[0]
    first_two = " ".join(first_words[:2])
    
    return first_word in BAD_START_WORDS or first_two in BAD_START_WORDS

def clean_chunk_boundaries(chunks, min_len=80, max_len=450):
    cleaned = []
    
    for chunk in chunks:
        chunk = str(chunk).strip()
        if not chunk:
            continue
        
        if cleaned and (len(chunk) < min_len or starts_bad(chunk)):
            merged = cleaned[-1].rstrip() + " " + chunk.lstrip()
            if len(merged) <= max_len:
                cleaned[-1] = merged
            else:
                cleaned.append(chunk)
        else:
            cleaned.append(chunk)
    
    return cleaned

In [34]:
all_chunks = []

for text in retrieval_df["context"]:
    raw_chunks = chunk_text_by_paragraph(text, max_len=300, overlap=50, min_len=80)
    fixed_chunks = clean_chunk_boundaries(raw_chunks, min_len=80, max_len=450)
    all_chunks.extend(fixed_chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

print("Unique chunks after boundary cleaning:", chunks_df.shape)
chunks_df.head(15)

Unique chunks after boundary cleaning: (6196, 1)


,chunk_text
0,BAŞLANGIÇ [5]
1,Türk Vatanı ve Milletinin ebedi varlığını ve Y...
2,Dünya milletleri ailesinin eşit haklara sahip ...
3,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
4,"Kuvvetler ayrımının, Devlet organları arasında..."
5,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
6,ceği ve laiklik ilkesinin gereği olarak kutsal...
7,Her Türk vatandaşının bu Anayasadaki temel hak...
8,Topluca Türk vatandaşlarının milli gurur ve if...
9,arşılıklı içten sevgi ve kardeşlik duygularıyl...


In [35]:
import re
import pandas as pd

In [36]:
def normalize_legal_text(text):
    text = str(text)

    text = text.replace("\\n", "\n")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r" *\n *", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [37]:
def split_legal_text_into_blocks(text):
    text = normalize_legal_text(text)

    parts = re.split(r"\n\s*\n", text)

    cleaned_parts = []
    for part in parts:
        part = part.strip()
        if not part:
            continue

        cleaned_parts.append(part)

    return cleaned_parts

In [38]:
def split_long_block(block, max_len=500):
    block = block.strip()

    if len(block) <= max_len:
        return [block]

    sentences = re.split(r"(?<=[.!?;:])\s+", block)

    chunks = []
    current = ""

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        if len(current) + len(sent) + 1 <= max_len:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)

            if len(sent) > max_len:
                words = sent.split()
                piece = ""
                for w in words:
                    if len(piece) + len(w) + 1 <= max_len:
                        piece = f"{piece} {w}".strip()
                    else:
                        if piece:
                            chunks.append(piece)
                        piece = w
                if piece:
                    current = piece
                else:
                    current = ""
            else:
                current = sent

    if current:
        chunks.append(current)

    return chunks

In [39]:
def build_legal_chunks(text, min_len=80, max_len=500):
    blocks = split_legal_text_into_blocks(text)

    merged_blocks = []
    buffer = ""

    for block in blocks:
        if not buffer:
            buffer = block
        elif len(buffer) < min_len:
            buffer = f"{buffer}\n\n{block}".strip()
        else:
            merged_blocks.append(buffer)
            buffer = block

    if buffer:
        merged_blocks.append(buffer)

    final_chunks = []
    for block in merged_blocks:
        final_chunks.extend(split_long_block(block, max_len=max_len))

    final_chunks = [c.strip() for c in final_chunks if c and len(c.strip()) >= 30]

    return final_chunks

In [40]:
all_chunks = []

for text in retrieval_df["context"]:
    chunks = build_legal_chunks(text, min_len=80, max_len=500)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame({"chunk_text": all_chunks})
chunks_df = chunks_df.drop_duplicates().reset_index(drop=True)

print("Unique chunks:", chunks_df.shape)
chunks_df.head(20)

Unique chunks: (4470, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,Dünya milletleri ailesinin eşit haklara sahip ...
2,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,"Kuvvetler ayrımının, Devlet organları arasında..."
4,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
5,Her Türk vatandaşının bu Anayasadaki temel hak...
6,Topluca Türk vatandaşlarının milli gurur ve if...
7,"FİKİR, İNANÇ VE KARARIYLA anlaşılmak, sözüne v..."
8,"TÜRK MİLLETİ TARAFINDAN, demokrasiye aşık Türk..."
9,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...


In [41]:
def looks_like_bad_start(text):
    text = str(text).strip()
    if not text:
        return True

    bad_prefixes = (
        "ve ", "veya ", "ile ", "da ", "de ", "ama ", "ancak ", "fakat ",
        "gibi ", "çünkü ", "ise ", "ceği ", "acağı ", "ını ", "ini ",
        "unu ", "ünü ", "arı ", "eri ", "arşılıklı "
    )

    lowered = text.lower()
    return lowered.startswith(bad_prefixes)

bad_chunks = chunks_df[chunks_df["chunk_text"].apply(looks_like_bad_start)]
print("Bad-looking chunk count:", len(bad_chunks))
bad_chunks.head(20)

Bad-looking chunk count: 10


,chunk_text
65,ve üzerime aldığım görevi tarafsızlıkla yerine...
2841,ancak yapılan ödemeler de geri istenemez. II. ...
3400,Ancak vekile\r\nyetki verildiği veya durumun z...
3467,"Ancak saklayan, öngörülemeyen durumlar dolayıs..."
3480,"Ancak işletenler, zararın saklatan veya ziyare..."
4188,"Ancak bu ceza, dörtte birinden dörtte üçüne ka..."
4257,"Ancak bu süreler, iklim, mevsim, o yerdeki gel..."
4409,Ancak iş arama iznini toplu kullanmak isteyen ...
4434,Ancak iş sözleşmesi belirli süreli olarak yapı...
4447,veya işçilerin toplu bulunduğu yerler gibi işç...


In [42]:
chunks_df = chunks_df[~chunks_df["chunk_text"].apply(looks_like_bad_start)].reset_index(drop=True)

print("Final clean chunks:", chunks_df.shape)
chunks_df.head(10)

Final clean chunks: (4460, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,Dünya milletleri ailesinin eşit haklara sahip ...
2,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,"Kuvvetler ayrımının, Devlet organları arasında..."
4,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."
5,Her Türk vatandaşının bu Anayasadaki temel hak...
6,Topluca Türk vatandaşlarının milli gurur ve if...
7,"FİKİR, İNANÇ VE KARARIYLA anlaşılmak, sözüne v..."
8,"TÜRK MİLLETİ TARAFINDAN, demokrasiye aşık Türk..."
9,ÜÇÜNCÜ KISIM\n\nCUMHURİYETİN TEMEL ORGANLARI\n...


In [43]:
chunks_df.to_csv("../data/processed/retrieval_chunks.csv", index=False, encoding="utf-8-sig")
print("Clean retrieval chunks saved.")

Clean retrieval chunks saved.


In [44]:
import pandas as pd

chunks_df = pd.read_csv("../data/processed/retrieval_chunks.csv")

if "chunk_id" not in chunks_df.columns:
    chunks_df = chunks_df.reset_index(drop=True)
    chunks_df["chunk_id"] = ["ctx_" + str(i).zfill(5) for i in range(len(chunks_df))]

cols = ["chunk_id", "chunk_text"] + [c for c in chunks_df.columns if c not in ["chunk_id", "chunk_text"]]
chunks_df = chunks_df[cols]

chunks_df.to_csv("../data/processed/retrieval_chunks_with_ids.csv", index=False, encoding="utf-8-sig")

print("chunks_df shape:", chunks_df.shape)
chunks_df.head()

chunks_df shape: (4460, 2)


,chunk_id,chunk_text
0,ctx_00000,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,ctx_00001,Dünya milletleri ailesinin eşit haklara sahip ...
2,ctx_00002,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,ctx_00003,"Kuvvetler ayrımının, Devlet organları arasında..."
4,ctx_00004,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."


In [45]:
eval_df = df_clean[["soru", "cevap", "context"]].copy().reset_index(drop=True)

eval_df = eval_df.sample(30, random_state=42).reset_index(drop=True)

print("Evaluation set shape:", eval_df.shape)
eval_df.head()

Evaluation set shape: (30, 3)


,soru,cevap,context
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...
1,Sözleşme devri işleminde devralan tarafın sözl...,"Sözleşme devri işleminde devralan taraf, sözle...",BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...
2,"Anayasa, hangi milli değerlerin korunmasını am...","Anayasa, Türk milli menfaatlerini, Türk varlığ...",BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...
3,Kişinin malını koruyamayacak durumda olması ve...,Kişinin malını koruyamayacak durumda olması ve...,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\r\n\r\nO...
4,Bilgi edinme başvurusu ile ilgili idari yargıy...,"Evet, bilgi edinme başvurusu ile ilgili idari ...",ÜÇÜNCÜ BÖLÜM\r\n\r\nBilgi Edinme Başvurusu\r\n...


In [47]:
print("len(chunks_df):", len(chunks_df))
print("chunk_embeddings shape:", chunk_embeddings.shape)
print("len(chunk_tensor):", len(chunk_tensor))
print("chunks_df columns:", chunks_df.columns.tolist())

len(chunks_df): 4460
chunk_embeddings shape: (5811, 384)
len(chunk_tensor): 5811
chunks_df columns: ['chunk_id', 'chunk_text']


In [48]:
import pandas as pd

chunks_df = pd.read_csv("../data/processed/retrieval_chunks.csv")
print("Loaded chunks:", chunks_df.shape)
chunks_df.head()

Loaded chunks: (4460, 1)


,chunk_text
0,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,Dünya milletleri ailesinin eşit haklara sahip ...
2,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,"Kuvvetler ayrımının, Devlet organları arasında..."
4,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."


In [49]:
if "chunk_id" not in chunks_df.columns:
    chunks_df = chunks_df.reset_index(drop=True)
    chunks_df["chunk_id"] = ["ctx_" + str(i).zfill(5) for i in range(len(chunks_df))]

chunks_df = chunks_df[["chunk_id", "chunk_text"]]
print(chunks_df.shape)
chunks_df.head()

(4460, 2)


,chunk_id,chunk_text
0,ctx_00000,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,ctx_00001,Dünya milletleri ailesinin eşit haklara sahip ...
2,ctx_00002,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,ctx_00003,"Kuvvetler ayrımının, Devlet organları arasında..."
4,ctx_00004,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."


In [50]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", chunk_embeddings.shape)
print("len(chunks_df):", len(chunks_df))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4995.49it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 140/140 [02:03<00:00,  1.13it/s]

Embeddings shape: (4460, 384)
len(chunks_df): 4460


In [51]:
import torch
chunk_tensor = torch.tensor(chunk_embeddings)
print("len(chunk_tensor):", len(chunk_tensor))

len(chunk_tensor): 4460


In [52]:
eval_df = df_clean[["soru", "cevap", "context"]].copy().reset_index(drop=True)
eval_df = eval_df.sample(30, random_state=42).reset_index(drop=True)

print(eval_df.shape)
eval_df.head()

(30, 3)


,soru,cevap,context
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...
1,Sözleşme devri işleminde devralan tarafın sözl...,"Sözleşme devri işleminde devralan taraf, sözle...",BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...
2,"Anayasa, hangi milli değerlerin korunmasını am...","Anayasa, Türk milli menfaatlerini, Türk varlığ...",BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...
3,Kişinin malını koruyamayacak durumda olması ve...,Kişinin malını koruyamayacak durumda olması ve...,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\r\n\r\nO...
4,Bilgi edinme başvurusu ile ilgili idari yargıy...,"Evet, bilgi edinme başvurusu ile ilgili idari ...",ÜÇÜNCÜ BÖLÜM\r\n\r\nBilgi Edinme Başvurusu\r\n...


In [53]:
from sentence_transformers import util
import numpy as np

gold_context_embeddings = embedding_model.encode(
    eval_df["context"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

gold_tensor = torch.tensor(gold_context_embeddings)

gold_chunk_ids = []
gold_chunk_texts = []
gold_match_scores = []

for i in range(len(eval_df)):
    sims = util.cos_sim(gold_tensor[i], chunk_tensor)[0].cpu().numpy()
    best_idx = int(np.argmax(sims))

    gold_chunk_ids.append(chunks_df.iloc[best_idx]["chunk_id"])
    gold_chunk_texts.append(chunks_df.iloc[best_idx]["chunk_text"])
    gold_match_scores.append(float(sims[best_idx]))

eval_df["gold_chunk_id"] = gold_chunk_ids
eval_df["gold_chunk_text"] = gold_chunk_texts
eval_df["gold_match_score"] = gold_match_scores

eval_df.head()

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


,soru,cevap,context,gold_chunk_id,gold_chunk_text,gold_match_score
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,ctx_03577,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,0.976433
1,Sözleşme devri işleminde devralan tarafın sözl...,"Sözleşme devri işleminde devralan taraf, sözle...",BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...,ctx_03007,BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...,0.987186
2,"Anayasa, hangi milli değerlerin korunmasını am...","Anayasa, Türk milli menfaatlerini, Türk varlığ...",BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,ctx_00000,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...,0.969214
3,Kişinin malını koruyamayacak durumda olması ve...,Kişinin malını koruyamayacak durumda olması ve...,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\r\n\r\nO...,ctx_03794,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\n\nONUNC...,0.987747
4,Bilgi edinme başvurusu ile ilgili idari yargıy...,"Evet, bilgi edinme başvurusu ile ilgili idari ...",ÜÇÜNCÜ BÖLÜM\r\n\r\nBilgi Edinme Başvurusu\r\n...,ctx_00769,ÜÇÜNCÜ BÖLÜM\n\nBilgi Edinme Başvurusu\n\nBaşv...,0.956717


In [54]:
print("len(chunks_df):", len(chunks_df))
print("chunk_tensor size:", chunk_tensor.shape[0])

assert len(chunks_df) == chunk_tensor.shape[0], "HATA: chunks_df ile chunk_embeddings aynı boyutta değil!"

len(chunks_df): 4460
chunk_tensor size: 4460


In [55]:
from sentence_transformers import util
import torch

gold_context_embeddings = embedding_model.encode(
    eval_df["context"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

chunk_tensor = torch.tensor(chunk_embeddings)
gold_tensor = torch.tensor(gold_context_embeddings)

gold_chunk_ids = []
gold_chunk_texts = []
gold_match_scores = []

for i in range(len(eval_df)):
    sims = util.cos_sim(gold_tensor[i], chunk_tensor)[0].cpu().numpy()
    best_idx = int(np.argmax(sims))
    
    gold_chunk_ids.append(chunks_df.iloc[best_idx]["chunk_id"])
    gold_chunk_texts.append(chunks_df.iloc[best_idx]["chunk_text"])
    gold_match_scores.append(float(sims[best_idx]))

eval_df["gold_chunk_id"] = gold_chunk_ids
eval_df["gold_chunk_text"] = gold_chunk_texts
eval_df["gold_match_score"] = gold_match_scores

eval_df.head()

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]


,soru,cevap,context,gold_chunk_id,gold_chunk_text,gold_match_score
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,ctx_03577,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,0.976433
1,Sözleşme devri işleminde devralan tarafın sözl...,"Sözleşme devri işleminde devralan taraf, sözle...",BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...,ctx_03007,BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...,0.987186
2,"Anayasa, hangi milli değerlerin korunmasını am...","Anayasa, Türk milli menfaatlerini, Türk varlığ...",BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,ctx_00000,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...,0.969214
3,Kişinin malını koruyamayacak durumda olması ve...,Kişinin malını koruyamayacak durumda olması ve...,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\r\n\r\nO...,ctx_03794,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\n\nONUNC...,0.987747
4,Bilgi edinme başvurusu ile ilgili idari yargıy...,"Evet, bilgi edinme başvurusu ile ilgili idari ...",ÜÇÜNCÜ BÖLÜM\r\n\r\nBilgi Edinme Başvurusu\r\n...,ctx_00769,ÜÇÜNCÜ BÖLÜM\n\nBilgi Edinme Başvurusu\n\nBaşv...,0.956717


In [56]:
for i in range(min(10, len(eval_df))):
    print("=" * 120)
    print("Soru:", eval_df.loc[i, "soru"])
    print("\nOrijinal Context (ilk 300 karakter):")
    print(eval_df.loc[i, "context"][:300])
    print("\nGold Chunk ID:", eval_df.loc[i, "gold_chunk_id"])
    print("Gold Match Score:", round(eval_df.loc[i, "gold_match_score"], 4))
    print("\nGold Chunk Text (ilk 300 karakter):")
    print(eval_df.loc[i, "gold_chunk_text"][:300])
    print()

Soru: Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?

Orijinal Context (ilk 300 karakter):
Ceza Kanununun amacı
MADDE 1. - (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar,

Gold Chunk ID: ctx_03577
Gold Match Score: 0.9764

Gold Chunk Text (ilk 300 karakter):
Ceza Kanununun amacı
MADDE 1. - (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar,

Soru: Sözleşme devri işleminde devralan tarafın sözleşmedeki durumu ne olur?

Orijinal Context (ilk 300 karakter):
BİRİNCİ KISIM
Genel Hükümler
BEŞİNCİ BÖLÜM
Borç İlişkilerinde Taraf Değişiklikleri
ÜÇÜNC

In [57]:
def retrieve_top_k_with_ids(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)

    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(dist)
        })
    return results

In [59]:
print("len(chunks_df):", len(chunks_df))
print("chunk_embeddings shape:", chunk_embeddings.shape)
print("FAISS index total:", index.ntotal)
print("chunks_df columns:", chunks_df.columns.tolist())

len(chunks_df): 4460
chunk_embeddings shape: (4460, 384)
FAISS index total: 5811
chunks_df columns: ['chunk_id', 'chunk_text']


In [60]:
import faiss
import numpy as np

print("Before rebuild:")
print("len(chunks_df):", len(chunks_df))
print("chunk_embeddings shape:", chunk_embeddings.shape)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype("float32"))

print("After rebuild:")
print("FAISS index total vectors:", index.ntotal)

Before rebuild:
len(chunks_df): 4460
chunk_embeddings shape: (4460, 384)
After rebuild:
FAISS index total vectors: 4460


In [61]:
def retrieve_top_k_with_ids(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_embedding, k)

    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        if idx < 0 or idx >= len(chunks_df):
            print(f"UYARI: idx={idx} ama len(chunks_df)={len(chunks_df)}")
            continue

        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(dist)
        })
    return results

In [62]:
print("len(chunks_df):", len(chunks_df))
print("chunk_embeddings:", chunk_embeddings.shape[0])
print("index.ntotal:", index.ntotal)

assert len(chunks_df) == chunk_embeddings.shape[0] == index.ntotal

len(chunks_df): 4460
chunk_embeddings: 4460
index.ntotal: 4460


In [63]:
all_results = []

for _, row in eval_df.iterrows():
    question = row["soru"]
    gold_chunk_id = row["gold_chunk_id"]
    
    retrieved = retrieve_top_k_with_ids(
        query=question,
        model=embedding_model,
        index=index,
        chunks_df=chunks_df,
        k=5
    )
    
    for item in retrieved:
        all_results.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "distance": item["distance"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_df = pd.DataFrame(all_results)

print("Results shape:", results_df.shape)
results_df.head(10)

Results shape: (150, 6)


,question,gold_chunk_id,rank,predicted_chunk_id,distance,predicted_chunk_text
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03577,1,ctx_01107,6.213552,a) Türk Ceza Kanununda yer alan;\n\n1. Örgüt f...
1,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03577,2,ctx_03734,6.813298,İKİNCİ BÖLÜM\n\nKanunun Uygulama Alanı\n\nGöre...
2,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03577,3,ctx_01484,7.349382,"c) Ulaşılan kanaat, sanığın suç oluşturduğu sa..."
3,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03577,4,ctx_03817,7.452248,"a) Türk kanunlarına göre suç değilse,\r\nb) Dü..."
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03577,5,ctx_04104,7.453783,Hükûmete karşı suç\r\nMADDE 312. - (1) Cebir v...
5,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03007,1,ctx_03008,5.737097,Sözleşmeyi devralan ile devreden arasında yapı...
6,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03007,2,ctx_03309,6.741226,Sözleşmenin fesih bildirimiyle sona ereceği ka...
7,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03007,3,ctx_03010,7.253783,"Sözleşmeye katılmanın geçerliliği, katılma kon..."
8,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03007,4,ctx_03070,7.316164,Sözleşmenin hüküm ve sonuçlarını doğurması ve ...
9,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03007,5,ctx_03307,7.434904,"Devir işlemiyle, devralan, bütün hak ve borçla..."


In [64]:
results_df.to_csv("../data/processed/retrieval_results_dense.csv", index=False, encoding="utf-8-sig")
eval_df.to_csv("../data/processed/eval_set_with_gold.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- ../data/processed/retrieval_results_dense.csv")
print("- ../data/processed/eval_set_with_gold.csv")

Saved:
- ../data/processed/retrieval_results_dense.csv
- ../data/processed/eval_set_with_gold.csv


In [65]:
import math

grouped = results_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(1 + 1)  
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

metrics_df = pd.DataFrame([
    {"Metric": k, "Value": round(v, 4)} for k, v in metrics.items()
])

metrics_df

,Metric,Value
0,Recall@1,0.0333
1,Recall@3,0.1000
2,Recall@5,0.1333
3,MRR@5,0.0750
4,NDCG@5,0.0898


In [66]:
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Recall@1: 0.0333
Recall@3: 0.1000
Recall@5: 0.1333
MRR@5: 0.0750
NDCG@5: 0.0898


In [67]:
sample_questions = eval_df["soru"].head(5).tolist()

for q in sample_questions:
    print("=" * 120)
    print("QUESTION:", q)
    
    temp = results_df[results_df["question"] == q].sort_values("rank")
    gold_id = temp["gold_chunk_id"].iloc[0]
    print("GOLD CHUNK ID:", gold_id)
    print()
    
    for _, row in temp.iterrows():
        print(f"Rank {row['rank']} | Predicted: {row['predicted_chunk_id']} | Distance: {row['distance']:.4f}")
        print(row["predicted_chunk_text"][:250])
        print("-" * 80)

QUESTION: Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?
GOLD CHUNK ID: ctx_03577

Rank 1 | Predicted: ctx_01107 | Distance: 6.2136
a) Türk Ceza Kanununda yer alan;

1. Örgüt faaliyeti çerçevesinde işlenip işlenmediğine bakılmaksızın uyuşturucu veya uyarıcı madde imal ve ticareti (madde 188),
--------------------------------------------------------------------------------
Rank 2 | Predicted: ctx_03734 | Distance: 6.8133
İKİNCİ BÖLÜM

Kanunun Uygulama Alanı

Görev suçları
MADDE 10. - (1) Yabancı ülkede Türkiye namına memuriyet veya görev üstlenmiş olup da bundan dolayı bir suç işleyen kimse, bu fiile ilişkin olarak yabancı ülkede hakkında mahkûmiyet hükmü verilmiş b
--------------------------------------------------------------------------------
Rank 3 | Predicted: ctx_01484 | Distance: 7.3494
c) Ulaşılan kanaat, sanığın suç oluşturduğu sabit görülen fiili ve bunun nitelendirilmesi; bu hususta ileri sürülen istemleri de dikkate alarak, Türk Ceza Kanununun 61 ve 62 nci m

In [68]:
summary_df = pd.DataFrame([{
    "Method": "Dense Retrieval (FAISS + MiniLM)",
    "Recall@1": round(metrics["Recall@1"], 4),
    "Recall@3": round(metrics["Recall@3"], 4),
    "Recall@5": round(metrics["Recall@5"], 4),
    "MRR@5": round(metrics["MRR@5"], 4),
    "NDCG@5": round(metrics["NDCG@5"], 4),
}])

summary_df

,Method,Recall@1,Recall@3,Recall@5,MRR@5,NDCG@5
0,Dense Retrieval (FAISS + MiniLM),0.0333,0.1,0.1333,0.075,0.0898


In [69]:
import faiss
import numpy as np

chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("Index total:", index.ntotal)
print("Chunk count:", len(chunks_df))

Batches: 100%|██████████| 140/140 [02:08<00:00,  1.09it/s]

Index total: 4460
Chunk count: 4460


In [70]:
def retrieve_top_k_with_ids(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx < 0 or idx >= len(chunks_df):
            continue

        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "score": float(score)
        })
    return results

In [71]:
chunks_df = pd.DataFrame({"chunk_text": all_chunks}).drop_duplicates().reset_index(drop=True)

if "chunk_id" not in chunks_df.columns:
    chunks_df["chunk_id"] = ["ctx_" + str(i).zfill(5) for i in range(len(chunks_df))]

chunks_df = chunks_df[["chunk_id", "chunk_text"]]
print(chunks_df.shape)
chunks_df.head()

(4470, 2)


,chunk_id,chunk_text
0,ctx_00000,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1,ctx_00001,Dünya milletleri ailesinin eşit haklara sahip ...
2,ctx_00002,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
3,ctx_00003,"Kuvvetler ayrımının, Devlet organları arasında..."
4,ctx_00004,"Hiçbir faaliyetin Türk milli menfaatlerinin, T..."


In [72]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

Batches: 100%|██████████| 140/140 [02:03<00:00,  1.13it/s]


In [73]:
import faiss

faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("len(chunks_df):", len(chunks_df))
print("chunk_embeddings:", chunk_embeddings.shape[0])
print("index.ntotal:", index.ntotal)

len(chunks_df): 4470
chunk_embeddings: 4470
index.ntotal: 4470


In [74]:
eval_df = df_clean[["soru", "cevap", "context"]].copy().reset_index(drop=True)
eval_df = eval_df.sample(30, random_state=42).reset_index(drop=True)

print(eval_df.shape)

(30, 3)


In [75]:
from sentence_transformers import util
import torch
import numpy as np

gold_context_embeddings = embedding_model.encode(
    eval_df["context"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

faiss.normalize_L2(gold_context_embeddings)

chunk_tensor = torch.tensor(chunk_embeddings)
gold_tensor = torch.tensor(gold_context_embeddings)

gold_chunk_ids = []
gold_chunk_texts = []
gold_match_scores = []

for i in range(len(eval_df)):
    sims = util.cos_sim(gold_tensor[i], chunk_tensor)[0].cpu().numpy()
    best_idx = int(np.argmax(sims))

    gold_chunk_ids.append(chunks_df.iloc[best_idx]["chunk_id"])
    gold_chunk_texts.append(chunks_df.iloc[best_idx]["chunk_text"])
    gold_match_scores.append(float(sims[best_idx]))

eval_df["gold_chunk_id"] = gold_chunk_ids
eval_df["gold_chunk_text"] = gold_chunk_texts
eval_df["gold_match_score"] = gold_match_scores

eval_df.head()

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


,soru,cevap,context,gold_chunk_id,gold_chunk_text,gold_match_score
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,ctx_03582,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...,0.976433
1,Sözleşme devri işleminde devralan tarafın sözl...,"Sözleşme devri işleminde devralan taraf, sözle...",BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...,ctx_03009,BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...,0.987186
2,"Anayasa, hangi milli değerlerin korunmasını am...","Anayasa, Türk milli menfaatlerini, Türk varlığ...",BAŞLANGIÇ [5]\n\n \n\nTürk Vatanı ve Milletini...,ctx_00000,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...,0.969214
3,Kişinin malını koruyamayacak durumda olması ve...,Kişinin malını koruyamayacak durumda olması ve...,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\r\n\r\nO...,ctx_03799,İKİNCİ KISIM\r\nKişilere Karşı Suçlar\n\nONUNC...,0.987747
4,Bilgi edinme başvurusu ile ilgili idari yargıy...,"Evet, bilgi edinme başvurusu ile ilgili idari ...",ÜÇÜNCÜ BÖLÜM\r\n\r\nBilgi Edinme Başvurusu\r\n...,ctx_00770,ÜÇÜNCÜ BÖLÜM\n\nBilgi Edinme Başvurusu\n\nBaşv...,0.956717


In [76]:
def retrieve_top_k_with_ids(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx < 0 or idx >= len(chunks_df):
            continue

        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "score": float(score)
        })
    return results

In [77]:
all_results = []

for _, row in eval_df.iterrows():
    question = row["soru"]
    gold_chunk_id = row["gold_chunk_id"]

    retrieved = retrieve_top_k_with_ids(
        query=question,
        model=embedding_model,
        index=index,
        chunks_df=chunks_df,
        k=5
    )

    for item in retrieved:
        all_results.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "score": item["score"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_df = pd.DataFrame(all_results)
results_df.head()

,question,gold_chunk_id,rank,predicted_chunk_id,score,predicted_chunk_text
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,1,ctx_01108,0.816768,a) Türk Ceza Kanununda yer alan;\n\n1. Örgüt f...
1,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,2,ctx_03739,0.796856,İKİNCİ BÖLÜM\n\nKanunun Uygulama Alanı\n\nGöre...
2,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,3,ctx_03822,0.780069,"a) Türk kanunlarına göre suç değilse,\r\nb) Dü..."
3,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,4,ctx_03742,0.779307,"- (1) Bir yabancı, 13 üncü maddede yazılı suçl..."
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,5,ctx_01485,0.778652,"c) Ulaşılan kanaat, sanığın suç oluşturduğu sa..."


In [78]:
import math

grouped = results_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(2)
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

Recall@1: 0.0333
Recall@3: 0.1000
Recall@5: 0.1333
MRR@5: 0.0750
NDCG@5: 0.0898


In [79]:
summary_df = pd.DataFrame([{
    "Method": "Dense Retrieval (Cosine/IP + New Chunking)",
    "Recall@1": round(metrics["Recall@1"], 4),
    "Recall@3": round(metrics["Recall@3"], 4),
    "Recall@5": round(metrics["Recall@5"], 4),
    "MRR@5": round(metrics["MRR@5"], 4),
    "NDCG@5": round(metrics["NDCG@5"], 4),
}])

summary_df

,Method,Recall@1,Recall@3,Recall@5,MRR@5,NDCG@5
0,Dense Retrieval (Cosine/IP + New Chunking),0.0333,0.1,0.1333,0.075,0.0898


In [80]:
for i in range(10):
    print("=" * 140)
    print("QUESTION:")
    print(eval_df.loc[i, "soru"])

    print("\nGOLD CHUNK ID:")
    print(eval_df.loc[i, "gold_chunk_id"])

    print("\nGOLD CHUNK TEXT:")
    print(eval_df.loc[i, "gold_chunk_text"][:500])

    print("\nORIGINAL CONTEXT:")
    print(eval_df.loc[i, "context"][:500])
    print()

QUESTION:
Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?

GOLD CHUNK ID:
ctx_03582

GOLD CHUNK TEXT:
Ceza Kanununun amacı
MADDE 1. - (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar, ceza ve güvenlik tedbirlerinin türleri düzenlenmiştir. Suçta ve cezada kanunîlik ilkesi
MADDE 2.

ORIGINAL CONTEXT:
Ceza Kanununun amacı
MADDE 1. - (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar, ceza ve güvenlik tedbirlerinin türleri düzenlenmiştir.
Suçta ve cezada kanunîlik ilkesi
MADDE 2. - (1) Kanunun açıkça suç saymadığı bir fiil için kimseye ceza verilem

In [81]:
manual_questions = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?"
]

for q in manual_questions:
    print("=" * 120)
    print("QUESTION:", q)
    results = retrieve_top_k_with_ids(q, embedding_model, index, chunks_df, k=10)

    for item in results:
        print(f"Rank {item['rank']} | {item['chunk_id']} | Score: {item['score']:.4f}")
        print(item["chunk_text"][:300])
        print("-" * 80)

QUESTION: Egemenlik kime aittir?
Rank 1 | ctx_00002 | Score: 0.7146
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;
--------------------------------------------------------------------------------
Rank 2 | ctx_02616 | Score: 0.6849
DÖRDÜNCÜ KİTAP
EŞYA HUKUKU
İKİNCİ KISIM
SINIRLI AYNÎ HAKLAR
BİRİNCİ BÖLÜM
İRTİFAK HAKLARI VE TAŞINMAZ YÜKÜ
ÜÇÜNCÜ AYIRIM
TAŞINMAZ YÜKÜ
A. Konusu
Madde 839- Taşınmaz yükü, bir taşınmazın malikini yalnız o taşınmazla sorumlu olmak 
üzere diğer bir kimseye bir şey vermek veya yapmakla yükümlü
--------------------------------------------------------------------------------
Rank 3 | ctx_02824 | Score: 0.6523
Tescil ve ilân Cumhurbaşkanınca çıkarılan yönetmelik hükümlerine göre yapılır.12
V. Mal ve hakların kazanılması ve sorumluluk
Madde 

In [82]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [str(text).lower().split() for text in chunks_df["chunk_text"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve_top_k(query, chunks_df, bm25, k=5):
    tokenized_query = str(query).lower().split()
    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "score": float(scores[idx])
        })
    return results

In [83]:
print(chunks_df.shape)
print(chunks_df.columns.tolist())

print(eval_df.shape)
print(eval_df.columns.tolist())

(4470, 2)
['chunk_id', 'chunk_text']
(30, 6)
['soru', 'cevap', 'context', 'gold_chunk_id', 'gold_chunk_text', 'gold_match_score']


In [84]:
from rank_bm25 import BM25Okapi
import numpy as np

tokenized_corpus = [
    str(text).lower().split()
    for text in chunks_df["chunk_text"].tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 corpus size:", len(tokenized_corpus))

BM25 corpus size: 4470


In [85]:
def bm25_retrieve_top_k(query, chunks_df, bm25, k=5):
    tokenized_query = str(query).lower().split()
    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "score": float(scores[idx])
        })

    return results

In [86]:
test_questions = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?"
]

for q in test_questions:
    print("=" * 120)
    print("QUESTION:", q)

    results = bm25_retrieve_top_k(q, chunks_df, bm25, k=5)

    for item in results:
        print(f"Rank {item['rank']} | {item['chunk_id']} | Score: {item['score']:.4f}")
        print(item["chunk_text"][:300])
        print("-" * 80)

QUESTION: Egemenlik kime aittir?
Rank 1 | ctx_01790 | Score: 9.1722
8. Cumhurbaşkanına hakaret (madde 299),

9. Devletin egemenlik alametlerini aşağılama (madde 300),
--------------------------------------------------------------------------------
Rank 2 | ctx_04078 | Score: 8.1191
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler

ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
Rank 3 | ctx_00358 | Score: 8.1191
VI. Egemenlik

Madde 6 – Egemenlik, kayıtsız şartsız Milletindir.

Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
Rank 4 | ctx_00875 | Score: 7.9957
(2) Birinci fıkra uyarınca yapılabilen incelemeler, bulunan ve kime ait olduğu belli olmayan beden parçaları üzerinde de yapılabilir. Birinci fıkranın ikinci cümlesi, bu 

In [87]:
all_results_bm25 = []

for _, row in eval_df.iterrows():
    question = row["soru"]
    gold_chunk_id = row["gold_chunk_id"]

    retrieved = bm25_retrieve_top_k(
        query=question,
        chunks_df=chunks_df,
        bm25=bm25,
        k=5
    )

    for item in retrieved:
        all_results_bm25.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "score": item["score"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_bm25_df = pd.DataFrame(all_results_bm25)

print(results_bm25_df.shape)
results_bm25_df.head()

(150, 6)


,question,gold_chunk_id,rank,predicted_chunk_id,score,predicted_chunk_text
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,1,ctx_02859,11.994899,Türk Bayrağının ve özel bayrakların standartla...
1,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,2,ctx_00065,9.925908,ve üzerime aldığım görevi tarafsızlıkla yerine...
2,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,3,ctx_01838,9.536952,g) Türk Ceza Kanununun 61 inci maddesindeki sı...
3,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,4,ctx_00836,9.499419,Kapsam ve Tanımlar\n\nKanunun kapsamı\n\nMadde...
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,5,ctx_02873,9.360089,Bayrak direğinin konulması\r\nMadde 7 – Bayrak...


In [88]:
import math

grouped = results_bm25_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(2)
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

bm25_metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

for k, v in bm25_metrics.items():
    print(f"{k}: {v:.4f}")

Recall@1: 0.0333
Recall@3: 0.2667
Recall@5: 0.3333
MRR@5: 0.1428
NDCG@5: 0.1903


In [89]:
bm25_summary_df = pd.DataFrame([{
    "Method": "BM25 Retrieval",
    "Recall@1": round(bm25_metrics["Recall@1"], 4),
    "Recall@3": round(bm25_metrics["Recall@3"], 4),
    "Recall@5": round(bm25_metrics["Recall@5"], 4),
    "MRR@5": round(bm25_metrics["MRR@5"], 4),
    "NDCG@5": round(bm25_metrics["NDCG@5"], 4),
}])

bm25_summary_df

,Method,Recall@1,Recall@3,Recall@5,MRR@5,NDCG@5
0,BM25 Retrieval,0.0333,0.2667,0.3333,0.1428,0.1903


In [90]:
def min_max_normalize(arr):
    arr = np.array(arr, dtype=float)
    if arr.max() == arr.min():
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())

In [91]:
def hybrid_retrieve_top_k(query, model, index, chunks_df, bm25, k=5, alpha=0.6):
    
    # Dense
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {int(idx): float(score) for idx, score in zip(dense_indices, dense_scores)}

    # BM25
    tokenized_query = str(query).lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Normalize
    dense_arr = np.array([dense_score_map.get(i, 0.0) for i in range(len(chunks_df))])
    bm25_arr = np.array(bm25_scores)

    dense_norm = min_max_normalize(dense_arr)
    bm25_norm = min_max_normalize(bm25_arr)

    # Combine
    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    # Top-k
    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "score": float(final_scores[idx])
        })

    return results

In [92]:
for q in test_questions:
    print("="*100)
    print("QUESTION:", q)

    results = hybrid_retrieve_top_k(q, embedding_model, index, chunks_df, bm25, k=5)

    for r in results:
        print(f"{r['rank']} | {r['chunk_id']} | {r['score']:.4f}")
        print(r["chunk_text"][:200])
        print("-"*50)

QUESTION: Egemenlik kime aittir?
1 | ctx_00358 | 0.9054
VI. Egemenlik

Madde 6 – Egemenlik, kayıtsız şartsız Milletindir.

Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------
2 | ctx_04078 | 0.8463
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler

ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------
3 | ctx_01790 | 0.7211
8. Cumhurbaşkanına hakaret (madde 299),

9. Devletin egemenlik alametlerini aşağılama (madde 300),
--------------------------------------------------
4 | ctx_03800 | 0.6717
a) Kime ait olursa olsun kamu kurum ve kuruluşlarında veya ibadete ayrılmış yerlerde bulunan ya da kamu yararına veya hizmetine tahsis edilen eşya hakkında, b) Herkesin girebileceği bir yerde bırakılm
--------------------------------------------------
5 | ctx_02788 | 0.6543
İyiniyetli olmayan zilyet, şeyi k

In [94]:
import pandas as pd
import math

all_results_hybrid = []

for _, row in eval_df.iterrows():
    question = row["soru"]
    gold_chunk_id = row["gold_chunk_id"]

    retrieved = hybrid_retrieve_top_k(
        query=question,
        model=embedding_model,
        index=index,
        chunks_df=chunks_df,
        bm25=bm25,
        k=5,
        alpha=0.6
    )

    for item in retrieved:
        all_results_hybrid.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "score": item["score"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_hybrid_df = pd.DataFrame(all_results_hybrid)

print("Hybrid results shape:", results_hybrid_df.shape)
results_hybrid_df.head(10)

Hybrid results shape: (150, 6)


,question,gold_chunk_id,rank,predicted_chunk_id,score,predicted_chunk_text
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,1,ctx_01108,0.859286,a) Türk Ceza Kanununda yer alan;\n\n1. Örgüt f...
1,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,2,ctx_01649,0.847628,a) Soruşturulması ve kovuşturulması şikâyete b...
2,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,3,ctx_01599,0.841913,a) Türk Ceza Kanununda yer alan;\n\n1. Hakkı o...
3,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,4,ctx_00921,0.821827,a) Toplumsal olaylar sırasında işlenen cebir v...
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_03582,5,ctx_01838,0.799959,g) Türk Ceza Kanununun 61 inci maddesindeki sı...
5,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03009,1,ctx_03010,0.934602,Sözleşmeyi devralan ile devreden arasında yapı...
6,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03009,2,ctx_03009,0.893717,BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...
7,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03009,3,ctx_02990,0.810595,Yasal veya yargısal devir ve etkisi\r\nMADDE 1...
8,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03009,4,ctx_02988,0.732512,BİRİNCİ KISIM\r\nGenel Hükümler\r\nBEŞİNCİ BÖL...
9,Sözleşme devri işleminde devralan tarafın sözl...,ctx_03009,5,ctx_03308,0.732287,"Yukarıdaki hükümlere göre devir hâlinde, devir..."


In [95]:
grouped = results_hybrid_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(2)   # ideal rank = 1
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

hybrid_metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

for k, v in hybrid_metrics.items():
    print(f"{k}: {v:.4f}")

Recall@1: 0.0333
Recall@3: 0.2667
Recall@5: 0.3000
MRR@5: 0.1289
NDCG@5: 0.1716


In [96]:
hybrid_summary_df = pd.DataFrame([{
    "Method": "Hybrid Retrieval (Dense + BM25)",
    "Recall@1": round(hybrid_metrics["Recall@1"], 4),
    "Recall@3": round(hybrid_metrics["Recall@3"], 4),
    "Recall@5": round(hybrid_metrics["Recall@5"], 4),
    "MRR@5": round(hybrid_metrics["MRR@5"], 4),
    "NDCG@5": round(hybrid_metrics["NDCG@5"], 4),
}])

hybrid_summary_df

,Method,Recall@1,Recall@3,Recall@5,MRR@5,NDCG@5
0,Hybrid Retrieval (Dense + BM25),0.0333,0.2667,0.3,0.1289,0.1716


In [97]:
for i in range(10):
    print("=" * 140)
    print("QUESTION:")
    print(eval_df.loc[i, "soru"])

    print("\nGOLD CHUNK ID:")
    print(eval_df.loc[i, "gold_chunk_id"])

    print("\nGOLD CHUNK TEXT:")
    print(eval_df.loc[i, "gold_chunk_text"][:500])

    print("\nORIGINAL CONTEXT:")
    print(eval_df.loc[i, "context"][:500])
    print("\nGOLD MATCH SCORE:")
    print(eval_df.loc[i, "gold_match_score"])

QUESTION:
Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?

GOLD CHUNK ID:
ctx_03582

GOLD CHUNK TEXT:
Ceza Kanununun amacı
MADDE 1. - (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar, ceza ve güvenlik tedbirlerinin türleri düzenlenmiştir. Suçta ve cezada kanunîlik ilkesi
MADDE 2.

ORIGINAL CONTEXT:
Ceza Kanununun amacı
MADDE 1. - (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar, ceza ve güvenlik tedbirlerinin türleri düzenlenmiştir.
Suçta ve cezada kanunîlik ilkesi
MADDE 2. - (1) Kanunun açıkça suç saymadığı bir fiil için kimseye ceza verilem

In [99]:
manual_questions = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?",
    "Taşınmaz yükü nedir?",
    "Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?"
]

In [100]:
for q in manual_questions:
    print("=" * 140)
    print("QUESTION:", q)

    results = bm25_retrieve_top_k(q, chunks_df, bm25, k=10)

    for item in results:
        print(f"Rank {item['rank']} | {item['chunk_id']} | Score: {item['score']:.4f}")
        print(item["chunk_text"][:400])
        print("-" * 80)

QUESTION: Egemenlik kime aittir?
Rank 1 | ctx_01790 | Score: 9.1722
8. Cumhurbaşkanına hakaret (madde 299),

9. Devletin egemenlik alametlerini aşağılama (madde 300),
--------------------------------------------------------------------------------
Rank 2 | ctx_04078 | Score: 8.1191
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler

ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
Rank 3 | ctx_00358 | Score: 8.1191
VI. Egemenlik

Madde 6 – Egemenlik, kayıtsız şartsız Milletindir.

Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
Rank 4 | ctx_00875 | Score: 7.9957
(2) Birinci fıkra uyarınca yapılabilen incelemeler, bulunan ve kime ait olduğu belli olmayan beden parçaları üzerinde de yapılabilir. Birinci fıkranın ikinci cümlesi, bu 

In [104]:
manual_eval_df = pd.DataFrame([
    {
        "question": "Egemenlik kime aittir?",
        "gold_chunk_id": "ctx_00012",
        "gold_chunk_text": chunks_df[chunks_df["chunk_id"] == "ctx_00012"]["chunk_text"].iloc[0]
    }
])

In [105]:
all_results_manual_bm25 = []

for _, row in manual_eval_df.iterrows():
    question = row["question"]
    gold_chunk_id = row["gold_chunk_id"]

    retrieved = bm25_retrieve_top_k(
        query=question,
        chunks_df=chunks_df,
        bm25=bm25,
        k=5
    )

    for item in retrieved:
        all_results_manual_bm25.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "score": item["score"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_manual_bm25_df = pd.DataFrame(all_results_manual_bm25)
results_manual_bm25_df.head()

,question,gold_chunk_id,rank,predicted_chunk_id,score,predicted_chunk_text
0,Egemenlik kime aittir?,ctx_00012,1,ctx_01790,9.172234,"8. Cumhurbaşkanına hakaret (madde 299),\n\n9. ..."
1,Egemenlik kime aittir?,ctx_00012,2,ctx_04078,8.119081,DÖRDÜNCÜ KISIM\r\nMillete ve Devlete Karşı Suç...
2,Egemenlik kime aittir?,ctx_00012,3,ctx_00358,8.119081,"VI. Egemenlik\n\nMadde 6 – Egemenlik, kayıtsız..."
3,Egemenlik kime aittir?,ctx_00012,4,ctx_00875,7.995696,(2) Birinci fıkra uyarınca yapılabilen incelem...
4,Egemenlik kime aittir?,ctx_00012,5,ctx_02260,6.823497,Değişikliklerin kütüğe geçirilmesi\r\nMadde 46...


In [106]:
import math

grouped = results_manual_bm25_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(2)
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

manual_bm25_metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

for k, v in manual_bm25_metrics.items():
    print(f"{k}: {v:.4f}")

Recall@1: 0.0000
Recall@3: 0.0000
Recall@5: 0.0000
MRR@5: 0.0000
NDCG@5: 0.0000


In [107]:
print("manual_eval_df shape:", manual_eval_df.shape)
print(manual_eval_df.head())

print("\nchunks_df shape:", chunks_df.shape)
print(chunks_df.head())

print("\nmanual gold ids sample:")
print(manual_eval_df["gold_chunk_id"].tolist()[:10])

print("\nchunk ids sample:")
print(chunks_df["chunk_id"].tolist()[:10])

manual_eval_df shape: (1, 3)
                 question gold_chunk_id  \
0  Egemenlik kime aittir?     ctx_00012   

                                     gold_chunk_text  
0  B. Kanunların teklif edilmesi ve görüşülmesi\n...  

chunks_df shape: (4470, 2)
    chunk_id                                         chunk_text
0  ctx_00000  BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
1  ctx_00001  Dünya milletleri ailesinin eşit haklara sahip ...
2  ctx_00002  Millet iradesinin mutlak üstünlüğü, egemenliği...
3  ctx_00003  Kuvvetler ayrımının, Devlet organları arasında...
4  ctx_00004  Hiçbir faaliyetin Türk milli menfaatlerinin, T...

manual gold ids sample:
['ctx_00012']

chunk ids sample:
['ctx_00000', 'ctx_00001', 'ctx_00002', 'ctx_00003', 'ctx_00004', 'ctx_00005', 'ctx_00006', 'ctx_00007', 'ctx_00008', 'ctx_00009']


In [108]:
manual_gold_ids = set(manual_eval_df["gold_chunk_id"].astype(str).str.strip())
chunk_ids = set(chunks_df["chunk_id"].astype(str).str.strip())

missing_ids = manual_gold_ids - chunk_ids

print("manual_eval_df içindeki toplam unique gold id:", len(manual_gold_ids))
print("chunks_df içinde bulunan unique chunk id:", len(chunk_ids))
print("chunks_df içinde bulunamayan gold id sayısı:", len(missing_ids))
print("Eksik gold id'ler:", list(missing_ids)[:20])

manual_eval_df içindeki toplam unique gold id: 1
chunks_df içinde bulunan unique chunk id: 4470
chunks_df içinde bulunamayan gold id sayısı: 0
Eksik gold id'ler: []


In [109]:
print(manual_eval_df[["question", "gold_chunk_id"]])

                 question gold_chunk_id
0  Egemenlik kime aittir?     ctx_00012


In [111]:
manual_eval_df = pd.DataFrame([
    {"question": "Egemenlik kime aittir?", "gold_chunk_id": "ctx_00012"},
    {"question": "Türkiye Cumhuriyetinin yönetim şekli nedir?", "gold_chunk_id": "ctx_00105"},
    {"question": "Cumhurbaşkanının görevleri nelerdir?", "gold_chunk_id": "ctx_00422"},
    {"question": "Taşınmaz yükü nedir?", "gold_chunk_id": "ctx_01234"},
])

In [112]:
manual_questions = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?",
    "Taşınmaz yükü nedir?",
    "Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?"
]

In [113]:
for q in manual_questions:
    print("=" * 140)
    print("QUESTION:", q)

    results = bm25_retrieve_top_k(q, chunks_df, bm25, k=10)

    for item in results:
        print(f"Rank {item['rank']} | {item['chunk_id']} | Score: {item['score']:.4f}")
        print(item["chunk_text"][:350])
        print("-" * 80)

QUESTION: Egemenlik kime aittir?
Rank 1 | ctx_01790 | Score: 9.1722
8. Cumhurbaşkanına hakaret (madde 299),

9. Devletin egemenlik alametlerini aşağılama (madde 300),
--------------------------------------------------------------------------------
Rank 2 | ctx_04078 | Score: 8.1191
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler

ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
Rank 3 | ctx_00358 | Score: 8.1191
VI. Egemenlik

Madde 6 – Egemenlik, kayıtsız şartsız Milletindir.

Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
Rank 4 | ctx_00875 | Score: 7.9957
(2) Birinci fıkra uyarınca yapılabilen incelemeler, bulunan ve kime ait olduğu belli olmayan beden parçaları üzerinde de yapılabilir. Birinci fıkranın ikinci cümlesi, bu 

In [114]:
manual_eval_df = pd.DataFrame([
    {"question": "Egemenlik kime aittir?", "gold_chunk_id": "ctx_01790"},
    {"question": "Türkiye Cumhuriyetinin yönetim şekli nedir?", "gold_chunk_id": "ctx_00072"},
    {"question": "Cumhurbaşkanının görevleri nelerdir?", "gold_chunk_id": "ctx_00160"},
    {"question": "Taşınmaz yükü nedir?", "gold_chunk_id": "ctx_02619"},
    {"question": "Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?", "gold_chunk_id": "ctx_02859"}
])

manual_eval_df

,question,gold_chunk_id
0,Egemenlik kime aittir?,ctx_01790
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00072
2,Cumhurbaşkanının görevleri nelerdir?,ctx_00160
3,Taşınmaz yükü nedir?,ctx_02619
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_02859


In [115]:
print(manual_eval_df["gold_chunk_id"].nunique())
print(manual_eval_df)

5
                                            question gold_chunk_id
0                             Egemenlik kime aittir?     ctx_01790
1        Türkiye Cumhuriyetinin yönetim şekli nedir?     ctx_00072
2               Cumhurbaşkanının görevleri nelerdir?     ctx_00160
3                               Taşınmaz yükü nedir?     ctx_02619
4  Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...     ctx_02859


In [116]:
print(manual_eval_df)

                                            question gold_chunk_id
0                             Egemenlik kime aittir?     ctx_01790
1        Türkiye Cumhuriyetinin yönetim şekli nedir?     ctx_00072
2               Cumhurbaşkanının görevleri nelerdir?     ctx_00160
3                               Taşınmaz yükü nedir?     ctx_02619
4  Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...     ctx_02859


In [117]:
import pandas as pd
import math

all_results_manual_bm25 = []

for _, row in manual_eval_df.iterrows():
    question = row["question"]
    gold_chunk_id = row["gold_chunk_id"]

    retrieved = bm25_retrieve_top_k(
        query=question,
        chunks_df=chunks_df,
        bm25=bm25,
        k=5
    )

    for item in retrieved:
        all_results_manual_bm25.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "score": item["score"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_manual_bm25_df = pd.DataFrame(all_results_manual_bm25)

print("Manual BM25 results shape:", results_manual_bm25_df.shape)
results_manual_bm25_df.head(10)

Manual BM25 results shape: (25, 6)


,question,gold_chunk_id,rank,predicted_chunk_id,score,predicted_chunk_text
0,Egemenlik kime aittir?,ctx_01790,1,ctx_01790,9.172234,"8. Cumhurbaşkanına hakaret (madde 299),\n\n9. ..."
1,Egemenlik kime aittir?,ctx_01790,2,ctx_04078,8.119081,DÖRDÜNCÜ KISIM\r\nMillete ve Devlete Karşı Suç...
2,Egemenlik kime aittir?,ctx_01790,3,ctx_00358,8.119081,"VI. Egemenlik\n\nMadde 6 – Egemenlik, kayıtsız..."
3,Egemenlik kime aittir?,ctx_01790,4,ctx_00875,7.995696,(2) Birinci fıkra uyarınca yapılabilen incelem...
4,Egemenlik kime aittir?,ctx_01790,5,ctx_02260,6.823497,Değişikliklerin kütüğe geçirilmesi\r\nMadde 46...
5,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00072,1,ctx_00072,14.724831,Yabancı devletlere Türkiye Cumhuriyetinin tems...
6,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00072,2,ctx_00001,11.359140,Dünya milletleri ailesinin eşit haklara sahip ...
7,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00072,3,ctx_00351,11.246100,BİRİNCİ KISIM\n\nGENEL ESASLAR\n\nI. Devletin ...
8,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00072,4,ctx_00000,10.151783,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
9,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00072,5,ctx_00380,9.069137,b) Cumhurbaşkanının istemi ve tespit edeceği s...


In [118]:
grouped = results_manual_bm25_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(2)
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

manual_bm25_metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

for k, v in manual_bm25_metrics.items():
    print(f"{k}: {v:.4f}")

Recall@1: 1.0000
Recall@3: 1.0000
Recall@5: 1.0000
MRR@5: 1.0000
NDCG@5: 1.0000


In [119]:
manual_bm25_summary_df = pd.DataFrame([{
    "Method": "BM25 Retrieval (Manual Eval Set)",
    "Recall@1": round(manual_bm25_metrics["Recall@1"], 4),
    "Recall@3": round(manual_bm25_metrics["Recall@3"], 4),
    "Recall@5": round(manual_bm25_metrics["Recall@5"], 4),
    "MRR@5": round(manual_bm25_metrics["MRR@5"], 4),
    "NDCG@5": round(manual_bm25_metrics["NDCG@5"], 4),
}])

manual_bm25_summary_df

,Method,Recall@1,Recall@3,Recall@5,MRR@5,NDCG@5
0,BM25 Retrieval (Manual Eval Set),1.0,1.0,1.0,1.0,1.0


In [120]:
verified_questions = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?",
    "Taşınmaz yükü nedir?",
    "Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?"
]

verified_questions

['Egemenlik kime aittir?',
 'Türkiye Cumhuriyetinin yönetim şekli nedir?',
 'Cumhurbaşkanının görevleri nelerdir?',
 'Taşınmaz yükü nedir?',
 "Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?"]

In [121]:
verified_source_df = df_clean[df_clean["soru"].isin(verified_questions)][["soru", "context"]].copy().reset_index(drop=True)

print(verified_source_df.shape)
verified_source_df

(1, 2)


,soru,context
0,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Ceza Kanununun amacı\r\nMADDE 1. - (1) Ceza Ka...


In [122]:
verified_answers_df = pd.DataFrame([
    {
        "question": "Egemenlik kime aittir?",
        "answer_snippet": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "answer_snippet": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görevleri nelerdir?",
        "answer_snippet": "Cumhurbaşkanı Devletin başıdır."
    },
    {
        "question": "Taşınmaz yükü nedir?",
        "answer_snippet": "Taşınmaz yükü, bir taşınmazın malikini"
    },
    {
        "question": "Türk Ceza Kanunu'nda 'yargı görevi yapan' tanımı nasıl yapılır?",
        "answer_snippet": "Yargı görevi yapan deyiminden"
    }
])

verified_answers_df

,question,answer_snippet
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.
2,Cumhurbaşkanının görevleri nelerdir?,Cumhurbaşkanı Devletin başıdır.
3,Taşınmaz yükü nedir?,"Taşınmaz yükü, bir taşınmazın malikini"
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Yargı görevi yapan deyiminden


In [123]:
from sentence_transformers import util
import torch
import numpy as np

snippet_embeddings = embedding_model.encode(
    verified_answers_df["answer_snippet"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

faiss.normalize_L2(snippet_embeddings)

chunk_tensor = torch.tensor(chunk_embeddings)
snippet_tensor = torch.tensor(snippet_embeddings)

gold_chunk_ids = []
gold_chunk_texts = []
gold_scores = []

for i in range(len(verified_answers_df)):
    sims = util.cos_sim(snippet_tensor[i], chunk_tensor)[0].cpu().numpy()
    best_idx = int(np.argmax(sims))

    gold_chunk_ids.append(chunks_df.iloc[best_idx]["chunk_id"])
    gold_chunk_texts.append(chunks_df.iloc[best_idx]["chunk_text"])
    gold_scores.append(float(sims[best_idx]))

verified_answers_df["gold_chunk_id"] = gold_chunk_ids
verified_answers_df["gold_chunk_text"] = gold_chunk_texts
verified_answers_df["gold_match_score"] = gold_scores

verified_answers_df

Batches: 100%|██████████| 1/1 [00:00<00:00, 21.96it/s]


,question,answer_snippet,gold_chunk_id,gold_chunk_text,gold_match_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,ctx_00002,"Millet iradesinin mutlak üstünlüğü, egemenliği...",0.811011
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,ctx_00067,"Cumhurbaşkanı, Devlet başkanı sıfatıyla Türkiy...",0.750884
2,Cumhurbaşkanının görevleri nelerdir?,Cumhurbaşkanı Devletin başıdır.,ctx_00119,Cumhurbaşkanınca atanan Genelkurmay Başkanı; S...,0.773633
3,Taşınmaz yükü nedir?,"Taşınmaz yükü, bir taşınmazın malikini",ctx_02465,"Eklenti, asıl şey malikinin anlaşılabilen arzu...",0.731662
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,Yargı görevi yapan deyiminden,ctx_00844,i) Malen sorumlu: Yargılama konusu işin hükme ...,0.709112


In [124]:
for _, row in verified_answers_df.iterrows():
    print("=" * 140)
    print("QUESTION:")
    print(row["question"])

    print("\nANSWER SNIPPET:")
    print(row["answer_snippet"])

    print("\nGOLD CHUNK ID:")
    print(row["gold_chunk_id"])

    print("\nGOLD CHUNK TEXT:")
    print(row["gold_chunk_text"][:600])

    print("\nMATCH SCORE:")
    print(round(row["gold_match_score"], 4))

QUESTION:
Egemenlik kime aittir?

ANSWER SNIPPET:
Egemenlik kayıtsız şartsız Milletindir.

GOLD CHUNK ID:
ctx_00002

GOLD CHUNK TEXT:
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;

MATCH SCORE:
0.811
QUESTION:
Türkiye Cumhuriyetinin yönetim şekli nedir?

ANSWER SNIPPET:
Türkiye Devleti bir Cumhuriyettir.

GOLD CHUNK ID:
ctx_00067

GOLD CHUNK TEXT:
Cumhurbaşkanı, Devlet başkanı sıfatıyla Türkiye Cumhuriyetini ve Türk Milletinin birliğini temsil eder; Anayasanın uygulanmasını, Devlet organlarının düzenli ve uyumlu çalışmasını temin eder.

MATCH SCORE:
0.7509
QUESTION:
Cumhurbaşkanının görevleri nelerdir?

ANSWER SNIPPET:
Cumhurbaşkanı Devletin başıdır.

GOLD CHUNK ID:
ctx_00119

GOLD CHUNK TEXT:
Cumhurbaşkanınca atanan Genelkurmay Başkanı; Silahlı Kuvvetlerin k

In [125]:
verified_eval_df = verified_answers_df[["question", "gold_chunk_id"]].copy()
verified_eval_df

,question,gold_chunk_id
0,Egemenlik kime aittir?,ctx_00002
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00067
2,Cumhurbaşkanının görevleri nelerdir?,ctx_00119
3,Taşınmaz yükü nedir?,ctx_02465
4,Türk Ceza Kanunu'nda 'yargı görevi yapan' tanı...,ctx_00844


In [126]:
import pandas as pd
import math

all_results_verified_bm25 = []

for _, row in verified_eval_df.iterrows():
    question = row["question"]
    gold_chunk_id = row["gold_chunk_id"]

    retrieved = bm25_retrieve_top_k(
        query=question,
        chunks_df=chunks_df,
        bm25=bm25,
        k=5
    )

    for item in retrieved:
        all_results_verified_bm25.append({
            "question": question,
            "gold_chunk_id": gold_chunk_id,
            "rank": item["rank"],
            "predicted_chunk_id": item["chunk_id"],
            "score": item["score"],
            "predicted_chunk_text": item["chunk_text"]
        })

results_verified_bm25_df = pd.DataFrame(all_results_verified_bm25)

print(results_verified_bm25_df.shape)
results_verified_bm25_df.head(10)

(25, 6)


,question,gold_chunk_id,rank,predicted_chunk_id,score,predicted_chunk_text
0,Egemenlik kime aittir?,ctx_00002,1,ctx_01790,9.172234,"8. Cumhurbaşkanına hakaret (madde 299),\n\n9. ..."
1,Egemenlik kime aittir?,ctx_00002,2,ctx_04078,8.119081,DÖRDÜNCÜ KISIM\r\nMillete ve Devlete Karşı Suç...
2,Egemenlik kime aittir?,ctx_00002,3,ctx_00358,8.119081,"VI. Egemenlik\n\nMadde 6 – Egemenlik, kayıtsız..."
3,Egemenlik kime aittir?,ctx_00002,4,ctx_00875,7.995696,(2) Birinci fıkra uyarınca yapılabilen incelem...
4,Egemenlik kime aittir?,ctx_00002,5,ctx_02260,6.823497,Değişikliklerin kütüğe geçirilmesi\r\nMadde 46...
5,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00067,1,ctx_00072,14.724831,Yabancı devletlere Türkiye Cumhuriyetinin tems...
6,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00067,2,ctx_00001,11.359140,Dünya milletleri ailesinin eşit haklara sahip ...
7,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00067,3,ctx_00351,11.246100,BİRİNCİ KISIM\n\nGENEL ESASLAR\n\nI. Devletin ...
8,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00067,4,ctx_00000,10.151783,BAŞLANGIÇ [5]\n\nTürk Vatanı ve Milletinin ebe...
9,Türkiye Cumhuriyetinin yönetim şekli nedir?,ctx_00067,5,ctx_00380,9.069137,b) Cumhurbaşkanının istemi ve tespit edeceği s...


In [127]:
grouped = results_verified_bm25_df.groupby("question")

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0
mrr_at_5 = 0
ndcg_at_5 = 0

num_questions = len(grouped)

for question, group in grouped:
    group = group.sort_values("rank")
    gold_id = group["gold_chunk_id"].iloc[0]

    top1 = group[group["rank"] <= 1]["predicted_chunk_id"].tolist()
    top3 = group[group["rank"] <= 3]["predicted_chunk_id"].tolist()
    top5 = group[group["rank"] <= 5]["predicted_chunk_id"].tolist()

    if gold_id in top1:
        recall_at_1 += 1
    if gold_id in top3:
        recall_at_3 += 1
    if gold_id in top5:
        recall_at_5 += 1

    rr = 0
    dcg = 0

    for _, row in group[group["rank"] <= 5].iterrows():
        rank = int(row["rank"])
        pred_id = row["predicted_chunk_id"]

        if pred_id == gold_id:
            rr = 1 / rank
            dcg = 1 / math.log2(rank + 1)
            break

    idcg = 1 / math.log2(2)
    ndcg = dcg / idcg if idcg > 0 else 0

    mrr_at_5 += rr
    ndcg_at_5 += ndcg

verified_bm25_metrics = {
    "Recall@1": recall_at_1 / num_questions,
    "Recall@3": recall_at_3 / num_questions,
    "Recall@5": recall_at_5 / num_questions,
    "MRR@5": mrr_at_5 / num_questions,
    "NDCG@5": ndcg_at_5 / num_questions,
}

for k, v in verified_bm25_metrics.items():
    print(f"{k}: {v:.4f}")

Recall@1: 0.0000
Recall@3: 0.0000
Recall@5: 0.0000
MRR@5: 0.0000
NDCG@5: 0.0000


In [128]:
verified_bm25_summary_df = pd.DataFrame([{
    "Method": "BM25 Retrieval (Verified Eval Set)",
    "Recall@1": round(verified_bm25_metrics["Recall@1"], 4),
    "Recall@3": round(verified_bm25_metrics["Recall@3"], 4),
    "Recall@5": round(verified_bm25_metrics["Recall@5"], 4),
    "MRR@5": round(verified_bm25_metrics["MRR@5"], 4),
    "NDCG@5": round(verified_bm25_metrics["NDCG@5"], 4),
}])

verified_bm25_summary_df

,Method,Recall@1,Recall@3,Recall@5,MRR@5,NDCG@5
0,BM25 Retrieval (Verified Eval Set),0.0,0.0,0.0,0.0,0.0


In [129]:
for _, row in verified_eval_df.iterrows():
    question = row["question"]
    gold_chunk_id = row["gold_chunk_id"]

    print("=" * 140)
    print("QUESTION:", question)
    print("GOLD CHUNK ID:", gold_chunk_id)
    print()

    gold_match = chunks_df[chunks_df["chunk_id"] == gold_chunk_id]
    if len(gold_match) > 0:
        print("GOLD CHUNK TEXT:")
        print(gold_match["chunk_text"].iloc[0][:400])
    else:
        print("Gold chunk bulunamadı!")
    print()

    print("BM25 TOP-5:")
    results = bm25_retrieve_top_k(question, chunks_df, bm25, k=5)

    for item in results:
        mark = "<<< EXACT MATCH" if item["chunk_id"] == gold_chunk_id else ""
        print(f"Rank {item['rank']} | {item['chunk_id']} | Score: {item['score']:.4f} {mark}")
        print(item["chunk_text"][:300])
        print("-" * 80)

QUESTION: Egemenlik kime aittir?
GOLD CHUNK ID: ctx_00002

GOLD CHUNK TEXT:
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;

BM25 TOP-5:
Rank 1 | ctx_01790 | Score: 9.1722 
8. Cumhurbaşkanına hakaret (madde 299),

9. Devletin egemenlik alametlerini aşağılama (madde 300),
--------------------------------------------------------------------------------
Rank 2 | ctx_04078 | Score: 8.1191 
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler

ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
Rank 3 | ctx_00358 | Score: 8.1191 
VI. Egemenlik

Madde 6 – Egemenlik, kayıtsız şartsız Milletindir.

Türk Milleti, egemenliğini, Anayasanın koyduğu 

In [130]:
query_text = "Egemenlik kayıtsız şartsız Milletindir"

matches = chunks_df[chunks_df["chunk_text"].str.contains(query_text, case=False, na=False)]
print(matches[["chunk_id", "chunk_text"]].head(10))

Empty DataFrame
Columns: [chunk_id, chunk_text]
Index: []


In [47]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4760.52it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
sample_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].head(5).tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Sample embeddings shape:", sample_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.95it/s]

Sample embeddings shape: (5, 384)


In [49]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 140/140 [01:46<00:00,  1.32it/s]

Embeddings shape: (4460, 384)


In [50]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index total vectors:", index.ntotal)

FAISS index total vectors: 4460


In [51]:
def retrieve_top_k(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "chunk_text": chunks_df.iloc[idx]["chunk_text"],
            "distance": float(dist)
        })
    return results

In [52]:
queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görev ve yetkileri nelerdir?"
]

for q in queries:
    print(f"\nSORGUN: {q}\n")
    results = retrieve_top_k(q, embedding_model, index, chunks_df, k=3)

    for i, item in enumerate(results, 1):
        print(f"{i}. Distance: {item['distance']:.4f}")
        print(item["chunk_text"][:500])
        print("-" * 100)


SORGUN: Egemenlik kime aittir?

1. Distance: 11.4850
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;
----------------------------------------------------------------------------------------------------
2. Distance: 12.6890
DÖRDÜNCÜ KİTAP
EŞYA HUKUKU
İKİNCİ KISIM
SINIRLI AYNÎ HAKLAR
BİRİNCİ BÖLÜM
İRTİFAK HAKLARI VE TAŞINMAZ YÜKÜ
ÜÇÜNCÜ AYIRIM
TAŞINMAZ YÜKÜ
A. Konusu
Madde 839- Taşınmaz yükü, bir taşınmazın malikini yalnız o taşınmazla sorumlu olmak 
üzere diğer bir kimseye bir şey vermek veya yapmakla yükümlü kılar. Hak sahibi olarak, bir başka taşınmazın maliki de gösterilebilir.
----------------------------------------------------------------------------------------------------
3. Distance: 13.5992
Tescil ve ilân Cumhurbaşkanınca çıkarılan yönetmelik hükümler

In [53]:
faiss.write_index(index, "../data/processed/faiss_index.index")
np.save("../data/processed/chunk_embeddings.npy", chunk_embeddings)

print("FAISS index and embeddings saved.")

FAISS index and embeddings saved.


In [54]:
test_queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görevleri nelerdir?"
]

for q in test_queries:
    print(f"\nQUERY: {q}\n")
    
    results = retrieve_top_k(q, embedding_model, index, chunks_df, k=3)
    
    for i, item in enumerate(results, 1):
        print(f"{i}. Distance: {item['distance']:.4f}")
        print(item["chunk_text"][:200])
        print("-" * 80)


QUERY: Egemenlik kime aittir?

1. Distance: 11.4850
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi 
--------------------------------------------------------------------------------
2. Distance: 12.6890
DÖRDÜNCÜ KİTAP
EŞYA HUKUKU
İKİNCİ KISIM
SINIRLI AYNÎ HAKLAR
BİRİNCİ BÖLÜM
İRTİFAK HAKLARI VE TAŞINMAZ YÜKÜ
ÜÇÜNCÜ AYIRIM
TAŞINMAZ YÜKÜ
A. Konusu
Madde 839- Taşınmaz yükü, bir taşınmazın malik
--------------------------------------------------------------------------------
3. Distance: 13.5992
Tescil ve ilân Cumhurbaşkanınca çıkarılan yönetmelik hükümlerine göre yapılır.12
V. Mal ve hakların kazanılması ve sorumluluk
Madde 105- Özgülenen malların mülkiyeti ile haklar, tüzel kişiliğin kaza
--------------------------------------------------------------------------------

QUERY: Türkiye Cumhuriyetinin yönetim şekli nedir?

1. Distance: 12.1